# Dataset analysis

## Configuration

In [ ]:
import sys
sys.path.append('../src')

Load machine specific configuration

In [ ]:
CONFIGURATION = '../configuration'

import configparser
import os
conf = configparser.ConfigParser()
conf.read((
    os.path.join(CONFIGURATION, f"default.conf"),
    os.path.join(CONFIGURATION, f"{os.uname().nodename}.conf")
))
conf.sections()

## Data Import

Preprocessed data

In [ ]:
SOURCE = '../.data/processed.pkl'

import pandas as pd
source = pd.read_pickle(SOURCE)
source = source[source['success']].drop(columns=['success']).reset_index(drop=True)
source

In [ ]:
tokens = set(source['token'].unique())
conditions = set(source['condition'].unique())
states = set(col for col in source.columns if col.startswith('state_'))

print(f"{len(tokens)} participants in {len(conditions)} conditions expressing {len(states)} states")

Action Units

In [ ]:
from utils.csv import csv_col_dict
ACTION_UNITS = csv_col_dict(
    conf['Features']['action-units'].strip().split('\n')
)['name']

Survey

In [ ]:
FILE_SURVEY = os.path.join(
    conf['Dataset']['path-root'],
    conf['Dataset']['file-survey']
)
survey_raw = pd.read_csv(FILE_SURVEY)
survey = survey_raw[survey_raw[survey_raw.columns[7]].isin(tokens)]
survey

## Demographics

In [ ]:
age = survey['SD01. Wie alt sind Sie?']
age.columns = ['Age']
sex = survey['SD03. Bitte geben Sie Ihr Geschlecht an.'].value_counts(normalize=True)
sex.index = ['male', 'female', 'n. a.', 'divers']
edu = survey['SD08. Was ist der höchste Bildungsabschluss, den Sie bereits erreicht haben?'].value_counts(normalize=True)
edu.index = ['High school', 'Bachelor', 'Master', 'Apprenticeship', 'Doctorate']

import matplotlib.pyplot as plt
import numpy as np
_, axs = plt.subplots(1,3, figsize=(12,4))
age.plot(kind='box', ax=axs[0], ylabel='')
sex.plot(kind='bar', ax=axs[1], ylabel='', xlabel='Gender')
edu.plot(kind='bar', ax=axs[2], ylabel='', xlabel='Education')
axs[0].set_xticklabels(['Age'])
plt.suptitle('Dataset Demographics')
plt.show()

## State Windows

In [ ]:
def annotate_window(recording):
    # sorted in tiem domain
    recording = recording.sort_values('timestamp')
    # identify state changes
    for state in states:
        state_change = recording[state].diff() != 0
        state_starts = recording['timestamp'][state_change].reset_index(drop=True)
        recording[state+'_period_id'] = state_change.cumsum()
        recording[state+'_period_begin'] = (recording[state+'_period_id'] - 1).map(state_starts)
        recording[state+'_period_end'] = recording[state+'_period_id'].map(state_starts)
        recording[state+'_period_time'] = recording[state+'_period_end'] - recording[state+'_period_begin']

    return recording

data_windows = pd.concat(
    pd.concat(
        # process per recording
        annotate_window(recording)
        for c, recording in participant.groupby('condition')
    )
    for p, participant in source.groupby('token')
)
data_windows

In [ ]:
states_a = ['state_confusion', 'state_understanding']
plt.boxplot([
    data_windows[data_windows[state]].groupby(
        ['token','condition',state+'_period_id']
    )[state+'_period_time'].first().dropna()
    for state in states_a
], showfliers=False)
plt.xticks(range(1, len(states_a)+1), [s[6:] for s in states_a])
plt.xlabel('Affective state')
plt.ylabel('Duration (s)')

plt.suptitle('State window durations')
plt.show()

In [ ]:
_, axs = plt.subplots(3,1, sharex=True, sharey=True)
bins = list(range(30))
for state, ax in zip (states, axs):
    data_windows[data_windows[state]].groupby(
        ['token','condition',state+'_period_id']
    )[state+'_period_time'].first().dropna().plot(
        kind='hist', ax=ax, bins=bins, density=True, ylabel=state[6:])
plt.xlabel('Duration (s)')
plt.suptitle('State window duration')
plt.show()

## Action Units

In [ ]:
import matplotlib.pyplot as plt

### Distribution

In [ ]:
ax = source[ACTION_UNITS].astype('float').plot(kind='box', showfliers=False)
ax.set_xticklabels(ACTION_UNITS, rotation=90)
plt.title('Overall AU expression distribution')
plt.show()

=> AU28 Trivial because no variance

In [ ]:
def plot_cross_domains(title:str, plot, figsize=(10,6)):
    _, axs = plt.subplots(2,2, figsize=figsize, sharex=True, sharey=True, tight_layout=True)
    for i, (c, ax) in enumerate(zip(('neutral', 'stress'), axs)):
        ax[0].set_ylabel(c)
        for s, a in zip(('understanding', 'confusion'), ax):
            plot(a, c, s)
            if i == 1:
                a.set_xlabel(s)
                a.tick_params(axis='x', rotation=90)

    plt.suptitle(title)
    plt.show()

def get_domain(df:pd.DataFrame, condition:str, state:str) -> pd.DataFrame:
    return df[(df['condition'] == condition) & df['state_'+state]]

def plot_distribution(ax, condition, state):
    data = get_domain(source, condition, state)[ACTION_UNITS].astype('float')
    ax = data.plot(kind='box', ax=ax, showfliers=False)

plot_cross_domains('AU expression distribution per domain', plot_distribution)

=> Show similar distribution patterns through domains

### Variances

In [ ]:
ax = source[ACTION_UNITS].var().plot(
    kind='bar', label='variance of full dataset'
)
ax = source.groupby('token')[ACTION_UNITS].var().mean().plot(
    kind='bar', ax=ax, label='mean of variances per participant',
    color='violet', alpha=.75
)
ax.set_xticklabels(ACTION_UNITS, rotation=90)
plt.title('Overall AU expression variance')
plt.legend()
plt.show()

=> Lower variance per participant.

In [ ]:
def plot_variances(ax, condition, state):
    data = source
    ax = data[ACTION_UNITS].var().plot(
        kind='bar', ax=ax, label='(reference) full set',
        color='darkgray', width=0.1
    )
    data = get_domain(source, condition, state)
    ax = data[ACTION_UNITS].var().plot(
        kind='bar', ax=ax, label='total domain',
        alpha=0.75, 
    )
    ax = data.groupby('token')[ACTION_UNITS].var().mean().plot(
        kind='bar', ax=ax, label='mean per participant',
        color='violet', alpha=.75
    )
    ax.legend()

plot_cross_domains('AU expression variance per domain', plot_variances)

=> In domain variance usually lower than cross domain variance;
Variance per participant lower than overall variance

=> Individual participant characteristics (per domain)

**Cross domain variances**
As foundation we take the mean of variances per participant denoted as $var_p$.
Action unit set is denotet by $x_{nu}$ with **n**eutral condition and **u**nderstanding state.

Cros state $mean(var_p(x_{nu} \cup x_{nc}), var_p(x_{su} \cup x_{sc}))$

Cros condition $mean(var_p(x_{nu} \cup x_{su}), var_p(x_{nc} \cup x_{sc}))$

Total $var_p(x_{nu} \cup x_{nc} \cup x_{su} \cup x_{sc})$

Individual $mean(var_p(x_{nu}), var_p(x_{nc}), var_p(x_{su}), var_p(x_{sc}))$

In [ ]:
def var_p(df:pd.DataFrame) -> pd.DataFrame:
    return df.groupby('token')[ACTION_UNITS].var().mean()
    
data_nu = get_domain(source, 'neutral', 'understanding')
data_su = get_domain(source, 'stress', 'understanding')
data_nc = get_domain(source, 'neutral', 'confusion')
data_sc = get_domain(source, 'stress', 'confusion')

total = var_p(pd.concat((data_nu, data_nc, data_su, data_sc)))
individual = pd.concat(
    (var_p(data_nu), var_p(data_nc), var_p(data_su), var_p(data_su)),
    axis=1
).T.mean()
x_state = pd.concat(
    (var_p(pd.concat((data_nu, data_nc))), var_p(pd.concat((data_su, data_sc)))),
    axis=1
).T.mean()
x_condi = pd.concat(
    (var_p(pd.concat((data_nu, data_su))), var_p(pd.concat((data_nc, data_sc)))),
    axis=1
).T.mean()

data = pd.concat((total, x_state, x_condi, individual), axis=1)
data.columns = ('total', 'x-state', 'x-condi', 'individual')

ax = data.plot(kind='bar')
plt.title('Mean variance per participant across domains')
plt.show()




=> All cross variances below total (significance test?)

=> indicator for domain specific expressions

=> wheter state or condision domain has lower variance varies throughout action units

=> Run correlation analysis to find prominet AUs

### Correlation

In [ ]:
def plot_corr(ax, condition, state):
    data = get_domain(source, condition, state)[ACTION_UNITS].corr(method='pearson')
    ax.matshow(data, cmap='coolwarm', vmin=-1, vmax=1)
    #ax.set_xticks(range(len(ACTION_UNITS['name'])), ACTION_UNITS['name'])
    #ax.set_yticks(range(len(ACTION_UNITS['name'])), ACTION_UNITS['name'])

plot_cross_domains('In-Domain Action Unit correlation', plot_corr, figsize=(7,7))

=> Overall similar correlation pattern

In [ ]:
_, axs = plt.subplots(2,2, figsize=(10,6), sharex=True, sharey=True, tight_layout=True)
rows = (('neutral', 'confusion'), ('stress', 'understanding'))
cols = (('neutral', 'understanding'), ('stress', 'confusion'))
for i, ((c_y,s_y), ax) in enumerate(zip(rows, axs)):
    ax[0].set_ylabel(f"{c_y}+{s_y}")
    for (c_x, s_x), a in zip(cols, ax):
        data_y = get_domain(source, c_y, s_y).groupby('token')[ACTION_UNITS].mean()
        data_x = get_domain(source, c_x, s_x).groupby('token')[ACTION_UNITS].mean()
        a.bar(range(len(tokens)), data_y.corrwith(data_x, axis=1))
        a.axhline(y=.8, c='red')
        if i == 1:
            a.set_xlabel(f"{c_x}+{s_x}")
            a.tick_params(axis='x', rotation=90)
plt.suptitle('Cross-Domain correlations of participants by AU expression')
plt.show()

=> Most participants share their individual AU expressions across domains

In [ ]:
_, axs = plt.subplots(2,2, figsize=(10,6), sharex=True, sharey=True, tight_layout=True)
rows = (('neutral', 'confusion'), ('stress', 'understanding'))
cols = (('neutral', 'understanding'), ('stress', 'confusion'))
for i, ((c_y,s_y), ax) in enumerate(zip(rows, axs)):
    ax[0].set_ylabel(f"{c_y}+{s_y}")
    for (c_x, s_x), a in zip(cols, ax):
        data_y = get_domain(source, c_y, s_y).groupby('token')[ACTION_UNITS].mean()
        data_x = get_domain(source, c_x, s_x).groupby('token')[ACTION_UNITS].mean()
        a.bar(ACTION_UNITS, data_y.corrwith(data_x))
        if i == 1:
            a.set_xlabel(f"{c_x}+{s_x}")
            a.tick_params(axis='x', rotation=90)
plt.suptitle('Cross-Domain correlations of mean AU expression per participant')
plt.show()

=> Differences in cross domain correlations

### Feature scoring



Prominent AUs for detection understanding/confusion requires low correlation through **state**-domain and high correlation through **condition**-domain

absolute correlation through state change: $c_s = \frac{|corr(x_{nu}, x_{nc})| + |corr(x_{su}, x_{sc})|}{2}$

absolute correlation through condition change: $c_c = \frac{|corr(x_{nu}, x_{su})| + |corr(x_{nc}, x_{sc})|}{2}$

$c_s$ should be small, $c_c$ should be high

Feature score: $s = \frac{1 - c_s + c_c}{2}$

This yields a feature score $s \in [0,1]$ with higher scores representing best features for understanding/confusion detection

In [ ]:
data_nu = get_domain(source, 'neutral', 'understanding').groupby('token')[ACTION_UNITS].mean()
data_su = get_domain(source, 'stress', 'understanding').groupby('token')[ACTION_UNITS].mean()
data_nc = get_domain(source, 'neutral', 'confusion').groupby('token')[ACTION_UNITS].mean()
data_sc = get_domain(source, 'stress', 'confusion').groupby('token')[ACTION_UNITS].mean()

corr_s = (data_nu.corrwith(data_nc).abs() + data_su.corrwith(data_sc).abs()) / 2
corr_c = (data_nu.corrwith(data_su).abs() + data_nc.corrwith(data_sc).abs()) / 2
score = (1 - corr_s + corr_c) / 2

data = pd.concat(
    (
        corr_s.rename('state'),
        corr_c.rename('condition'),
        score.rename('score')
    ), axis=1
)
#data.index = ACTION_UNITS['name']
data = data.sort_values('score')
_, axs = plt.subplots(2,2, sharex=True, sharey=True)
ax = data['state'].plot(kind='bar', ax=axs[0,0], title='State change correlation')
ax = data['condition'].plot(kind='bar', ax=axs[0,1], title='Condition change correlation')

ax = data[['state', 'condition']].plot(kind='bar', ax=axs[1,0], title='Comparison')

ax = data['score'].plot(kind='bar', ax=axs[1,1], title='Score')

plt.suptitle('Feature scoring')
plt.show()

print('Top features:')
print('-'*16)
for key in score.sort_values(ascending=False)[:5].index:
    print(key)

=> Top scores for AU26 (Jaw drop), AU45 (Blink), AU9 (Nose Wrinkler), AU02 (Outer Brow Raiser) and AU01 (Inner Brow Raiser)

In [ ]:
ax = data['state'].sort_values().plot(kind='bar', title='State change correlation')

## Frequency exploration

As AU45 (Blinking) truned out to be a prominent feature we can explore blink frequencies through domains

First compute blink durations per video (not per window) to get blink frequnecies

In [ ]:
blink_data = source.copy()

Compute frequency (blinks / min): ($60 / duration)$

In [ ]:
blink_data['blink_frequency'] = 60 / blink_data['blink_interval_time']
# mark frequnecies above 100 as invalid
blink_data.loc[blink_data['blink_frequency'] > 100, 'blink_frequency'] = pd.NA

Remove listening state from data

In [ ]:
blink_data_releveant = blink_data#[blink_data['state'] != 'listening']

In [ ]:
plt.boxplot(
    [
        g.groupby(['condition', 'blink_interval_id'])['blink_frequency'].mean().dropna() 
        for _, g in blink_data_releveant.groupby('token')
    ], 
    showfliers=False
)
plt.axhline(15, label='15 blinks/min')
plt.legend()
plt.title('Total blink frequencies')
plt.xlabel('participant')
plt.ylabel('blinks/min')
plt.xticks([])
plt.show()

In [ ]:
_,axs = plt.subplots(1,2, figsize=(12,6), sharey=True)
# Un-weighted
axs[0].boxplot(
    [
        g.groupby(['condition', 'blink_interval_id'])['blink_frequency'].mean().dropna()
        for _, g in blink_data_releveant.groupby('token')
    ], 
    showfliers=False
)

# Time-weigted
axs[1].boxplot(
    [
        g['blink_frequency'].dropna()
        for _, g in blink_data_releveant.groupby('token')
    ], 
    showfliers=False
)

axs[0].set_title('Unweighted (Interval based)')
axs[1].set_title('Time-weighted (Frame based)')
axs[0].set_ylabel('blinks/min')
for ax in axs:
    ax.set_xticks([])
    ax.set_xlabel('Participants')
    ax.axhline(15, label='15 blinks/min')
    ax.legend()
plt.suptitle('Total blink frequencies')
plt.show()

In [ ]:
def plot_freq(ax, condition, state):
    data = get_domain(blink_data_releveant, condition, state)
    ax.boxplot(
        [
            g.groupby(['condition', 'blink_interval_id'])['blink_frequency'].mean().dropna() 
            for _, g in data.groupby('token')],
        showfliers=False
    )
    ax.axhline(15)
    ax.set_xticks([])

plot_cross_domains('In domain blink frequency distribution', plot_freq)

Cross domain comparison

In [ ]:
from scipy import stats
_, axs = plt.subplots(2,2, figsize=(12,6), sharey='row')
# Median
ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['state_'+s]
        ].groupby('token')['blink_frequency'].mean().rename(s)
        for s in ('understanding', 'confusion')),
    axis=1
).plot(kind='bar', ax=axs[0,0])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['state_understanding']
    ].groupby('token')['blink_frequency'].mean(),
    blink_data_releveant[
        blink_data_releveant['state_confusion']
    ].groupby('token')['blink_frequency'].mean()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_ylabel('Mean')
ax.set_title('State domain')

ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['condition'] == c
        ].groupby('token')['blink_frequency'].mean().rename(c)
        for c in ('neutral', 'stress')),
    axis=1
).plot(kind='bar', ax=axs[0,1])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['condition'] == 'neutral'
    ].groupby('token')['blink_frequency'].mean(),
    blink_data_releveant[
        blink_data_releveant['condition'] == 'stress'
    ].groupby('token')['blink_frequency'].mean()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_title('Condition domain')

# Variance
ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['state_'+s]
        ].groupby('token')['blink_frequency'].std().rename(s)
        for s in ('understanding', 'confusion')),
    axis=1
).plot(kind='bar', ax=axs[1,0])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['state_understanding']
    ].groupby('token')['blink_frequency'].var(),
    blink_data_releveant[
        blink_data_releveant['state_confusion']
    ].groupby('token')['blink_frequency'].var()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_ylabel('Stadard deviation')

ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['condition'] == c
        ].groupby('token')['blink_frequency'].std().rename(c)
        for c in ('neutral', 'stress')),
    axis=1
).plot(kind='bar', ax=axs[1,1])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['condition'] == 'neutral'
    ].groupby('token')['blink_frequency'].var(),
    blink_data_releveant[
        blink_data_releveant['condition'] == 'stress'
    ].groupby('token')['blink_frequency'].var()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])

plt.suptitle('Blink frequnecy change per participant across domains')
plt.show()

=> (Significant?) increase in mean blink frequency per participant for confusion (state-domain)

Unweighted by duration

In [ ]:
from scipy import stats
_, axs = plt.subplots(2,2, figsize=(12,6), sharey='row')
# Median
ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['state_'+s]
        ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').mean().rename(s)
        for s in ('understanding', 'confusion')),
    axis=1
).plot(kind='bar', ax=axs[0,0])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['state_understanding']
    ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').mean(),
    blink_data_releveant[
        blink_data_releveant['state_confusion']
    ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').mean()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_ylabel('Mean')
ax.set_title('State domain')

ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['condition'] == c
        ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').mean().rename(c)
        for c in ('neutral', 'stress')),
    axis=1
).plot(kind='bar', ax=axs[0,1])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['condition'] == 'neutral'
    ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').mean(),
    blink_data_releveant[
        blink_data_releveant['condition'] == 'stress'
    ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').mean()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_title('Condition domain')

# Variance
ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['state_'+s]
        ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').std().rename(s)
        for s in ('understanding', 'confusion')),
    axis=1
).plot(kind='bar', ax=axs[1,0])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['state_understanding']
    ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').std(),
    blink_data_releveant[
        blink_data_releveant['state_confusion']
    ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').std()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_ylabel('Stadard deviation')

ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['condition'] == c
        ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').std().rename(c)
        for c in ('neutral', 'stress')),
    axis=1
).plot(kind='bar', ax=axs[1,1])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['condition'] == 'neutral'
    ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').std(),
    blink_data_releveant[
        blink_data_releveant['condition'] == 'stress'
    ].groupby(['token', 'condition', 'blink_interval_id'])['blink_frequency'].first().groupby('token').std()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])

plt.suptitle('Blink frequnecy change per participant across domains')
plt.show()